# SE-ResNeXt-50 YOLO-ROI — Test-Split Evaluation (No Training)

Loads `best_model.pth` from the existing trained run and evaluates on the **held-out test split** using the **YOLO-ROI view only** (your own yolov8 crop).

> **This notebook is already configured for the trained run.** `RUN_DIR` points at `…/seresnext50_32x4d_yolo_roi/2026-08-21_15-54-31_396374_UTC` (best robust_val=0.7204, test_roi qwk=0.6824 from the paired fine-tune). Just **Runtime → Run all**.

## 0. Setup

In [1]:
# ── Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── Imports ──────────────────────────────────────────────────────────
import json, random, sys
from datetime import datetime, timezone
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import (
    average_precision_score, cohen_kappa_score,
    confusion_matrix, mean_absolute_error,
    precision_recall_fscore_support, roc_auc_score
)
from sklearn.preprocessing import label_binarize
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm

Mounted at /content/drive


## 1. Configuration

| Parameter | Value | Notes |
|---|---|---|
| `INPUT_SIZE` | `384` | Square resize (px) |
| `BATCH_SIZE` | `128` | Inference batch size |
| `NUM_WORKERS` | `4` | Dataloader workers |
| `SEED` | `42` | Reproducibility seed |
| `ROI_TEST_ROOT` | `…/densenet121_yolo_square_roi_trainvaltest_v2/test` | YOLO-ROI cropped test images (your yolov8) |

**`RUN_DIR` already points at:** `…/seresnext50_32x4d_yolo_roi/2026-08-21_15-54-31_396374_UTC` (best robust_val=0.7204, test_roi qwk=0.6824). Just **Runtime → Run all**.

In [2]:
# ── Paths — UPDATE THESE ────────────────────────────────────────────
ROI_TEST_ROOT = Path(
    '/content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/'
    'derived/densenet121_yolo_square_roi_trainvaltest_v2/test'
)

# Location of the trained run's outputs (best_model.pth is expected here)
RUN_DIR = Path('/content/drive/MyDrive/Models/seresnext50_32x4d_yolo_roi/2026-08-21_15-54-31_396374_UTC')
best_checkpoint_path = RUN_DIR / 'best_model.pth'
print(f'Evaluating run: {RUN_DIR}')
print(f'Checkpoint:     {best_checkpoint_path}')
print(f'Test root (YOLO-ROI): {ROI_TEST_ROOT}')

# ── Model & training config (must match the original run) ───────────
SEED = 42
INPUT_SIZE = 384
BATCH_SIZE = 128
NUM_WORKERS = 4

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

Evaluating run: /content/drive/MyDrive/Models/seresnext50_32x4d_yolo_roi/2026-08-21_15-54-31_396374_UTC
Checkpoint:     /content/drive/MyDrive/Models/seresnext50_32x4d_yolo_roi/2026-08-21_15-54-31_396374_UTC/best_model.pth
Test root (YOLO-ROI): /content/drive/MyDrive/Datasets/KneeXrayData_Mendeley_v1/derived/densenet121_yolo_square_roi_trainvaltest_v2/test
Device: cuda


## 2. Test Dataset (YOLO-ROI only)

Single-view dataset reader for the **held-out test split** using **YOLO-cropped images** (your yolov8 detector output, stored in `densenet121_yolo_square_roi_trainvaltest_v2/test/`).

Each grade folder (0–4) is scanned for `.png` files; the integer subfolder name is used as the true label. The same val_transform preprocessing (CLAHE → SquarePad → Resize(384) → ImageNet normalization) is applied as in training.

In [3]:
rows = []
for grade in range(5):
    grade_dir = ROI_TEST_ROOT / str(grade)
    if not grade_dir.exists():
        print(f'WARN: missing {grade_dir}')
        continue
    for path in sorted(grade_dir.glob('*.png')):
        rows.append({'path': str(path), 'true_grade': grade})
test_frame = pd.DataFrame(rows)
print(f'Test samples (YOLO-ROI): {len(test_frame)}')
print(test_frame.groupby('true_grade').size().to_string())

Test samples (YOLO-ROI): 1656
true_grade
0    639
1    296
2    447
3    223
4     51


## 3. Preprocessing, Dataset & Model

Identical to the training notebook — needed here so inference matches exactly.

**Preprocessing pipeline:**
1. OpenCV CLAHE (L-channel clip=1.25, grid=8×8) — contrast enhancement
2. SquarePad — pad smaller side to match the larger side (zero-fill)
3. Resize to `INPUT_SIZE` × `INPUT_SIZE`
4. ImageNet normalisation

**Dataloader:** `test_roi_loader` — YOLO-ROI view of the held-out test split (no augmentation, no random sampling).

**Model:** custom head (`num_features → 5`) on the pretrained backbone.

In [4]:
class OpenCVCLAHE:
    def __call__(self, image_rgb):
        lab = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l = cv2.createCLAHE(clipLimit=1.25, tileGridSize=(8, 8)).apply(l)
        return cv2.cvtColor(cv2.merge((l, a, b)), cv2.COLOR_LAB2RGB)

class SquarePad:
    def __call__(self, image_rgb):
        h, w = image_rgb.shape[:2]
        side = max(h, w)
        top = (side - h) // 2
        left = (side - w) // 2
        return cv2.copyMakeBorder(
            image_rgb, top, side - h - top, left, side - w - left,
            cv2.BORDER_CONSTANT, value=(0, 0, 0))

normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

val_transform = transforms.Compose([
    OpenCVCLAHE(), SquarePad(), transforms.ToPILImage(),
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    normalize,
])

class ROITestDataset(Dataset):
    'Single-view dataset: reads YOLO-cropped test images from ROI_TEST_ROOT.'
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True)
        self.transform = transform
        self.labels = self.frame.true_grade.astype(int).tolist()

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        img = cv2.imread(row['path'])
        if img is None:
            raise IOError(f'Cannot read {index}: {row["path"]}')
        return self.transform(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)), int(row['true_grade']), row['path']


class SEResNeXt50Model(nn.Module):
    'SE-ResNeXt-50 with timm num_classes=0 backbone + custom nn.Linear classifier. '\
    'Use num_classes=0 (NOT features_only) to match the state_dict layout saved by '\
    'the training run (timm classification head stripped, bare conv backbone with '\
    'layer1..layer4 attributes).'
    def __init__(self):
        super().__init__()
        self.backbone = timm.create_model(
            'seresnext50_32x4d', pretrained=False, num_classes=0)
        channels = self.backbone.num_features
        self.classifier = nn.Linear(channels, 5)

    def forward(self, images):
        # timm num_classes=0 already global-pools to (B, channels)
        features = self.backbone(images)
        return self.classifier(features)


model = SEResNeXt50Model().to(DEVICE)
print(f'Total params: {sum(p.numel() for p in model.parameters()):,}')

test_roi_loader = DataLoader(
    ROITestDataset(test_frame, val_transform),
    batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True)
print(f'Test batches (YOLO-ROI): {len(test_roi_loader)}')

Total params: 25,521,141
Test batches (YOLO-ROI): 13


/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## 4. Enhanced Evaluators

Per-class P/R/F1/AP/AUC, confusion matrix, mAP, mAUC — no training here.

**Metrics computed:**
| Metric | Description |
|---|---|
| `accuracy` | Exact match accuracy |
| `qwk` | Quadratic Weighted Kappa |
| `mae` | Mean Absolute Error |
| `off1_acc` | Off-by-≤1 tolerance accuracy |
| `macro_precision/recall/f1` | Macro-averaged across 5 grades |
| `macro_ap` | Macro-average Precision-Recall score |
| `mAP` | Mean per-class Average Precision |
| `mAUC` | Mean per-class ROC AUC |
| `confusion_matrix` | 5×5 confusion matrix |
| `per_class.{p,r,f1,ap,auc,support}` | Per-grade breakdown |

In [5]:
GRADE_NAMES = ['0 - Normal', '1 - Doubtful', '2 - Mild', '3 - Moderate', '4 - Severe']
NUM_CLASSES = 5

def evaluate(loader, tag='Test'):
    model.eval()
    all_labels, all_preds, all_probas, all_paths = [], [], [], []
    with torch.inference_mode():
        for tensors, labels, paths in tqdm(loader, desc=f'[{tag}]'):
            tensors = tensors.to(DEVICE, non_blocking=True)
            logits = model(tensors).float()
            probas  = F.softmax(logits, dim=1).cpu().numpy()
            all_labels.extend(labels.numpy())
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_probas.extend(probas)
            all_paths.extend(paths)
    y_true  = np.asarray(all_labels).astype(int)
    y_pred  = np.asarray(all_preds).astype(int)
    y_proba = np.asarray(all_probas)
    y_onehot = label_binarize(y_true, classes=range(NUM_CLASSES))
    # ── Aggregate ──
    qwk = float(cohen_kappa_score(y_true, y_pred, weights='quadratic'))
    macro_f1, macro_pr, macro_re, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0)
    macro_ap = float(average_precision_score(y_onehot, y_proba, average='macro'))
    # ── Per-class AP + AUC ──
    per_class_ap, per_class_auc = [], []
    for c in range(NUM_CLASSES):
        per_class_ap.append(float(average_precision_score(y_onehot[:, c], y_proba[:, c])))
        try:
            per_class_auc.append(float(roc_auc_score(y_onehot[:, c], y_proba[:, c])))
        except ValueError:
            per_class_auc.append(float('nan'))
    mAP  = float(np.nanmean(per_class_ap))
    mAUC = float(np.nanmean([x for x in per_class_auc if not np.isnan(x)]))
    # ── Per-class P/R/F1 ──
    p_c, r_c, f_c, s_c = precision_recall_fscore_support(
        y_true, y_pred, labels=range(NUM_CLASSES), zero_division=0)
    return {
        'accuracy':       float(np.mean(y_true == y_pred)),
        'qwk':           qwk,
        'mae':           float(mean_absolute_error(y_true, y_pred)),
        'off1_acc':      float(np.mean(np.abs(y_true - y_pred) <= 1)),
        'macro_f1':      float(macro_f1),
        'macro_pr':      float(macro_pr),
        'macro_re':      float(macro_re),
        'macro_ap':      macro_ap,
        'mAP':           mAP,
        'mAUC':          mAUC,
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=range(NUM_CLASSES)).tolist(),
        'per_class': {
            'precision': [float(x) for x in p_c],
            'recall':    [float(x) for x in r_c],
            'f1':        [float(x) for x in f_c],
            'support':   [int(x)   for x in s_c],
            'ap':        per_class_ap,
            'auc':       per_class_auc,
        },
        'sample_paths': list(all_paths),
        'sample_count': int(len(all_labels)),
    }

## 5. Load Checkpoint & Run Test Evaluation

Loads `best_model.pth` saved by the training run and evaluates on the **held-out test split** using **YOLO-ROI view only** (your yolov8 detector output).

Expects `best_model.pth` to contain at minimum:
```python
{'model_state_dict': ..., 'selection': <QWK value>}
```

In [6]:
best_ckpt = torch.load(best_checkpoint_path, map_location=DEVICE, weights_only=False)
model.load_state_dict(best_ckpt['model_state_dict'])
print(f"Loaded: {best_checkpoint_path}  (robust={best_ckpt['selection']:.4f})\n")

test_m = evaluate(test_roi_loader, tag='Test/ROI')

Loaded: /content/drive/MyDrive/Models/seresnext50_32x4d_yolo_roi/2026-08-21_15-54-31_396374_UTC/best_model.pth  (robust=0.7204)



/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


[Test/ROI]:   0%|          | 0/13 [00:00<?, ?it/s]

## 6. Detailed Metrics Report — Test / YOLO-ROI

In [7]:
# ═══════════════════════════════════════════════════════════════════════
#  AGGREGATE METRICS — Test split, YOLO-ROI view
# ═══════════════════════════════════════════════════════════════════════
m = test_m
print('=' * 70)
print(f"{'Metric':<24s}  {'Test (YOLO-ROI)':>16s}")
print('-' * 70)
agg_rows = [
    ('Accuracy',           m['accuracy']),
    ('QWK',                m['qwk']),
    ('MAE',                m['mae']),
    ('Off-by-1 Accuracy',  m['off1_acc']),
    ('Macro Precision',    m['macro_pr']),
    ('Macro Recall',       m['macro_re']),
    ('Macro F1',           m['macro_f1']),
    ('Macro AP',           m['macro_ap']),
    ('mAP',                m['mAP']),
    ('mAUC',               m['mAUC']),
]
for name, v in agg_rows:
    print(f'{name:<24s}  {v:16.4f}')
print('=' * 70)

# ═══════════════════════════════════════════════════════════════════════
#  PER-CLASS METRICS
# ═══════════════════════════════════════════════════════════════════════
print(f"\n{'Per-Class Metrics — Test (YOLO-ROI)':^72}")
print('=' * 70)
print(f"{'Grade':<22s}  {'Prec':>6s}  {'Rec':>6s}  {'F1':>6s}  {'AP':>6s}  {'AUC':>6s}  {'N':>5s}")
print('-' * 70)
pc = m['per_class']
for i, name in enumerate(GRADE_NAMES):
    print(f"{name:<22s}  "
          f"{pc['precision'][i]:6.4f}  {pc['recall'][i]:6.4f}  {pc['f1'][i]:6.4f}  "
          f"{pc['ap'][i]:6.4f}  {pc['auc'][i]:6.4f}  {pc['support'][i]:5d}")
print('=' * 70)

# ═══════════════════════════════════════════════════════════════════════
#  CONFUSION MATRIX
# ═══════════════════════════════════════════════════════════════════════
cm = m['confusion_matrix']
print('\nConfusion Matrix — Test (YOLO-ROI):')
cw = 7
header = ' ' * cw + ''.join(f"{i:>{cw}d}" for i in range(NUM_CLASSES))
print(header)
for i, row in enumerate(cm):
    print(f"{i:>{cw}d}" + ''.join(f"{v:>{cw}d}" for v in row))

Metric                     Test (YOLO-ROI)
----------------------------------------------------------------------
Accuracy                            0.5507
QWK                                 0.6824
MAE                                 0.6099
Off-by-1 Accuracy                   0.8557
Macro Precision                     0.5849
Macro Recall                        0.5526
Macro F1                            0.5354
Macro AP                            0.6023
mAP                                 0.6023
mAUC                                0.8219

                  Per-Class Metrics — Test (YOLO-ROI)                   
Grade                     Prec     Rec      F1      AP     AUC      N
----------------------------------------------------------------------
0 - Normal              0.6303  0.7152  0.6701  0.6821  0.8088    639
1 - Doubtful            0.2824  0.2500  0.2652  0.2419  0.6463    296
2 - Mild                0.5344  0.3826  0.4459  0.5424  0.7406    447
3 - Moderate            0.5936 

## 7. Save Metrics

In [8]:
final_metrics = {
    'split': 'test',
    'view':  'yolo_roi',
    'checkpoint_path': str(best_checkpoint_path),
    'metrics': test_m,
}
with open(RUN_DIR / 'final_metrics_test_roi.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)
print(f'Saved: {RUN_DIR / "final_metrics_test_roi.json"}')

Saved: /content/drive/MyDrive/Models/seresnext50_32x4d_yolo_roi/2026-08-21_15-54-31_396374_UTC/final_metrics_test_roi.json
